# Instructional Fine-Tuning of LLaMA 3B for Named Entity Recognition (NER)
This Colab notebook fine-tunes a LLaMA 3B model using English data from the MultiNERD dataset. It demonstrates full fine-tuning and prepares the dataset in prompt-response format suitable for instruction-based training.

In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 59.7 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

# Load and filter English-only entries from MultiNERD
dataset = load_dataset("Babelscape/multinerd", verification_mode='no_checks')
dataset = dataset.filter(lambda x: x['lang'] == 'en')

# Split into train/val/test
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
dataset["validation"] = dataset["test"].train_test_split(test_size=0.5, seed=42)["train"]
dataset["test"] = dataset["test"].train_test_split(test_size=0.5, seed=42)["test"]

README.md: 0.00B [00:00, ?B/s]

train_de.jsonl:   0%|          | 0.00/32.7M [00:00<?, ?B/s]

train_en.jsonl:   0%|          | 0.00/37.9M [00:00<?, ?B/s]

train_es.jsonl:   0%|          | 0.00/46.0M [00:00<?, ?B/s]

train_fr.jsonl:   0%|          | 0.00/46.2M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/334800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/335986 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1339200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/167400 [00:00<?, ? examples/s]

Filter:   0%|          | 0/167993 [00:00<?, ? examples/s]

In [4]:
# Define label names from MultiNERD classes, extended to cover all tag IDs in the dataset
label_names = [
    "O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC",
    "B-MISC", "I-MISC", "B-PROD", "I-PROD", "B-EVENT", "I-EVENT",
    "B-MEDIA", "I-MEDIA", "B-WORK", "I-WORK",
    "B-LOCgeoun", "I-LOCgeoun", "B-LOCwater", "I-LOCwater", # Additional LOC types from MultiNERD documentation
    "B-ORGtrans", "I-ORGtrans", "B-ORGpol", "I-ORGpol", # Additional ORG types
    "B-PRODsoft", "I-PRODsoft", # Additional PROD types
    "B-EVENTnat", "I-EVENTnat", # Additional EVENT types
    "B-MEDIApub", "I-MEDIApub", # Additional MEDIA types
    "B-WORKart", "I-WORKart" # Additional WORK types
]

def decode_tags(example):
    example['ner_label_names'] = [label_names[i] for i in example['ner_tags']]
    return example

dataset = dataset.map(decode_tags)

Map:   0%|          | 0/105024 [00:00<?, ? examples/s]

Map:   0%|          | 0/13128 [00:00<?, ? examples/s]

Map:   0%|          | 0/13128 [00:00<?, ? examples/s]

In [5]:
# Convert tokens and labels into instruction-style prompt-response pairs
def create_prompt(example):
    sentence = " ".join(example['tokens'])
    entities = []
    current = []
    ent_type = None
    for token, tag in zip(example['tokens'], example['ner_label_names']):
        if tag == "O":
            if current:
                entities.append(f"{' '.join(current)} ({ent_type})")
                current = []
                ent_type = None
        else:
            prefix, typ = tag.split("-")
            if prefix == "B":
                if current:
                    entities.append(f"{' '.join(current)} ({ent_type})")
                current = [token]
                ent_type = typ
            elif prefix == "I" and typ == ent_type:
                current.append(token)
            else:
                if current:
                    entities.append(f"{' '.join(current)} ({ent_type})")
                current = [token]
                ent_type = typ
    if current:
        entities.append(f"{' '.join(current)} ({ent_type})")
    prompt = f"Identify named entities in the sentence:\nSentence: {sentence}\nEntities:"
    response = ", ".join(entities) if entities else "None"
    return {"prompt": prompt, "response": response}

for split in ["train", "validation", "test"]:
    dataset[split] = dataset[split].map(create_prompt)

Map:   0%|          | 0/105024 [00:00<?, ? examples/s]

Map:   0%|          | 0/13128 [00:00<?, ? examples/s]

Map:   0%|          | 0/13128 [00:00<?, ? examples/s]

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load a smaller LLaMA-style model for demo purposes
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map='auto')

# Add a padding token to the tokenizer if it doesn't have one
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [7]:
# Merge prompt and response for causal LM fine-tuning
def format_for_lm(example):
    text = example['prompt'] + ' ' + example['response'] + tokenizer.eos_token
    return {"text": text}

tokenized_datasets = {}
for split in ["train", "validation"]:
    tokenized_datasets[split] = dataset[split].map(format_for_lm, remove_columns=dataset[split].column_names)

Map:   0%|          | 0/105024 [00:00<?, ? examples/s]

Map:   0%|          | 0/13128 [00:00<?, ? examples/s]

In [27]:
# Tokenize merged prompt-response pairs
def format_and_tokenize(examples):
    # Process each example in the batch
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for i in range(len(examples['prompt'])):
        prompt_text = " ".join(examples['prompt'][i]) if isinstance(examples['prompt'][i], list) else examples['prompt'][i]
        response_text = " ".join(examples['response'][i]) if isinstance(examples['response'][i], list) else examples['response'][i]
        text = prompt_text + ' ' + response_text + tokenizer.eos_token
        tokenized_inputs = tokenizer(text, truncation=True, padding="max_length", max_length=512)

        input_ids_list.append(tokenized_inputs["input_ids"])
        attention_mask_list.append(tokenized_inputs["attention_mask"])
        labels_list.append(tokenized_inputs["input_ids"].copy()) # Add labels for causal LM

    return {"input_ids": input_ids_list, "attention_mask": attention_mask_list, "labels": labels_list}


tokenized_datasets = {}
for split in ["train", "validation"]:
    tokenized_datasets[split] = dataset[split].map(
        format_and_tokenize,
        batched=True,
        remove_columns=dataset[split].column_names # remove original columns
    )

Map:   0%|          | 0/105024 [00:00<?, ? examples/s]

Map:   0%|          | 0/13128 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch

# Use standard DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="llama3b_ner_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

model.gradient_checkpointing_enable()
trainer.train()

/tmp/ipython-input-28-1036045679.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
